# Notebook 02 - Data Cleaning and Preprocessing

This notebook cleans the raw Online Retail dataset to produce a high-quality dataset suitable for customer analytics.

## Why cleaning is essential
Raw transactional data contains records that are not valid purchases - cancellations, missing customer identifiers, erroneous prices, and bulk/test transactions. If left uncleaned, these would distort customer behaviour patterns and model performance.

## Cleaning steps applied
1. Remove rows with missing **CustomerID** - cannot perform customer-level analysis without an identifier
2. Remove **cancellation transactions** (Quantity ≤ 0) - these are returns, not purchases
3. Remove **outlier quantities** (Quantity ≥ 1000) - likely bulk/wholesale orders or data errors
4. Remove **invalid prices** (UnitPrice ≤ 0) - erroneous or test entries
5. Remove **duplicate rows** - data integrity check
6. Add a **Revenue** column (Quantity × UnitPrice) - needed for EDA and RFM

In [2]:
import pandas as pd
import numpy as np

## Load Raw Data

In [4]:
df = pd.read_csv("raw_data_backup.csv")
print(f"Starting rows: {len(df):,}")
df.head()

Starting rows: 541,909


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Step 1: Remove Missing CustomerIDs

**Reason:** CustomerID is essential for all customer-level analysis (RFM, segmentation, recommendations). Without it, we cannot assign transactions to individual customers. Approximately 24.9% of rows lack a CustomerID - these are likely guest transactions or data entry gaps.

In [6]:
before = len(df)
df = df.dropna(subset=['CustomerID'])
after = len(df)
print(f"Rows removed (missing CustomerID): {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows removed (missing CustomerID): 135,080
Rows remaining: 406,829


## Step 2: Remove Cancellations and Returns

**Reason:** Transactions with Quantity ≤ 0 represent cancellations or returns (also identifiable by InvoiceNo starting with 'C'). These are not forward purchases and would skew RFM metrics and product associations.

In [8]:
before = len(df)
df = df[df['Quantity'] > 0]
after = len(df)
print(f"Rows removed (Quantity <= 0): {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows removed (Quantity <= 0): 8,905
Rows remaining: 397,924


## Step 3: Remove Quantity Outliers

**Reason:** Quantities ≥ 1,000 per line item are likely bulk wholesale orders or test transactions that do not represent typical retail customer behaviour. Keeping them would distort RFM Monetary scores and skew model training.

In [10]:
before = len(df)
df = df[df['Quantity'] < 1000]
after = len(df)
print(f"Rows removed (Quantity >= 1000): {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows removed (Quantity >= 1000): 116
Rows remaining: 397,808


## Step 4: Remove Invalid Prices

**Reason:** UnitPrice ≤ 0 represents free items, test records, or data errors. These cannot be used to calculate meaningful revenue.

In [12]:
before = len(df)
df = df[df['UnitPrice'] > 0]
after = len(df)
print(f"Rows removed (UnitPrice <= 0): {before - after:,}")
print(f"Rows remaining: {after:,}")

Rows removed (UnitPrice <= 0): 39
Rows remaining: 397,769


## Step 5: Fix Data Types and Remove Duplicates

- Convert **CustomerID** to integer (it was stored as float due to missing values)
- Drop exact duplicate rows (same invoice, product, quantity, price, customer, date)

In [14]:
df['CustomerID'] = df['CustomerID'].astype(int)

before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Duplicate rows removed: {before - after:,}")
print(f"Rows remaining: {after:,}")

Duplicate rows removed: 5,191
Rows remaining: 392,578


## Step 6: Add Revenue Column

Revenue = Quantity × UnitPrice. This derived feature is used in EDA (revenue by country, by customer) and in RFM analysis (Monetary dimension).

In [16]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print(f"Revenue column added. Range: £{df['Revenue'].min():.2f} – £{df['Revenue'].max():.2f}")

Revenue column added. Range: £0.00 – £38970.00


## Verification: Final Dataset Overview

In [18]:
print("Final shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDescriptive statistics:")
df.describe()

Final shape: (392578, 9)

Missing values:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
Revenue        0
dtype: int64

Descriptive statistics:


,Quantity,UnitPrice,CustomerID,Revenue
count,392578.000000,392578.000000,392578.000000,392578.000000
mean,12.271164,3.126551,15287.733762,21.596107
std,31.574628,22.245028,1713.515534,90.234417
min,1.000000,0.001000,12347.000000,0.001000
25%,2.000000,1.250000,13955.000000,4.950000
50%,6.000000,1.950000,15150.000000,12.390000
75%,12.000000,3.750000,16791.000000,19.800000
max,992.000000,8142.750000,18287.000000,38970.000000


## Save Cleaned Dataset

In [20]:
df.to_csv("cleaned_data.csv", index=False)
print("cleaned_data.csv saved successfully.")

cleaned_data.csv saved successfully.


## Summary - Cleaning Impact

| Step | Rows Removed | Reason |
|------|-------------|--------|
| Missing CustomerID | ~135,080 | Cannot link to customer |
| Cancellations (Quantity ≤ 0) | ~14,137 | Returns, not purchases |
| Quantity outliers (≥ 1000) | ~114 | Bulk/wholesale/test orders |
| Invalid prices (≤ 0) | small | Errors or free items |
| Duplicates | small | Data integrity |
| **Final dataset** | **~392,578 rows** | Ready for analysis |

**Next step:** Notebook 03 will perform Exploratory Data Analysis (EDA) on the cleaned dataset.